# NeuroFly-Bee Perú — Demostración de inferencia de punta a punta (v23)

Este cuaderno **no entrena**: carga los pesos que produjo `01_entrenar_colab.ipynb`, verifica
su integridad (SHA-256), hace volar los controladores en escenarios de **prueba** y muestra
qué miran las redes (Grad-CAM en la mosca, Integrated Gradients en la abeja).

1. Suba la carpeta del código (`.zip`) o ponga la URL del repositorio.
2. Suba `neuroflybee_pesos.zip` (o `resultados_v24.zip`, que lo contiene).
3. Ejecute las celdas en orden. Funciona en CPU; con GPU es más rápido.

## 1. Obtener el código

In [ ]:
import glob, os, zipfile
REPO_URL = ""   # p. ej. https://github.com/<usuario>/neuroflybee.git ; vacío = subir el .zip
if REPO_URL:
    !git clone -q $REPO_URL neuroflybee
else:
    from google.colab import files
    for name in files.upload():
        zipfile.ZipFile(name).extractall('.')
root = os.path.dirname(sorted(glob.glob('**/pyproject.toml', recursive=True), key=len)[0])
os.chdir(root or '.')
print('Proyecto en', os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

## 2. Cargar y verificar los pesos
Acepta `neuroflybee_pesos.zip` (sólo pesos) o el `.zip` de resultados completo del cuaderno 01.
`package_weights.py verify` compara tamaño y SHA-256 con `checkpoints/MANIFEST.json`.

In [ ]:
from google.colab import files
for name in files.upload():
    with zipfile.ZipFile(name) as zf:
        members = [m for m in zf.namelist() if m.startswith('checkpoints/') and not m.endswith('/')]
        if not members:
            raise SystemExit(f'{name} no contiene la carpeta checkpoints/')
        zf.extractall('.', members)          # el .zip de pesos y el de resultados la contienen
print(sorted(os.listdir('checkpoints')))
!python scripts/package_weights.py verify

## 3. Volar en un escenario de prueba
Mismos escenarios para todos: A*, A* + ALV y, si hay pesos, mosca, abeja e híbrido. Las
semillas de prueba (20000–20999) nunca se usaron para entrenar ni elegir épocas.

In [ ]:
from IPython.display import Image, display
import pandas as pd
FAMILIA = 'combined'     # nominal | unknown_obstacles | dynamic | wind | gps_loss | combined
SEMILLA = 20003
!python scripts/demo_inference.py --familia $FAMILIA --semilla $SEMILLA
display(pd.read_csv(f'results/v23_demo_{FAMILIA}_{SEMILLA}.csv'))
display(Image(f'results/figures/v23_demo_{FAMILIA}_{SEMILLA}.png', width=1100))

In [ ]:
# Las seis condiciones con la misma semilla
for familia in ['nominal', 'unknown_obstacles', 'dynamic', 'wind', 'gps_loss', 'combined']:
    !python scripts/demo_inference.py --familia $familia --semilla $SEMILLA > /dev/null
    display(Image(f'results/figures/v23_demo_{familia}_{SEMILLA}.png', width=1100))

## 4. ¿Qué miran las redes? (R5)
Grad-CAM: qué zona del ojo de la mosca provocó cada maniobra. Integrated Gradients: qué
direcciones del horizonte usó la abeja, con la prueba de fidelidad por borrado.

In [ ]:
!python scripts/explain.py --episodes 5
for f in ['results/figures/v15_gradcam_mosca.png', 'results/figures/v15_ig_abeja.png']:
    if os.path.exists(f):
        display(Image(f, width=1100))
    else:
        print('No se generó', f, '(¿faltan pesos?)')

## 5. Visor HTML con los controladores neuronales (opcional)
Genera `results/visor/index.html` con los vuelos de los controladores para los que hay pesos.

In [ ]:
!python scripts/build_viewer.py
from google.colab import files
!cd results && zip -q -r ../visor_v23.zip visor
files.download('visor_v23.zip')